# Vesuvius Challenge - Surface Detection
## Submission Notebook — nnU-Net Pipeline

3D surface segmentation using nnU-Net v2 with 1st place post-processing.

**Model:** nnU-Net v2 (3d_fullres, 200 epochs, trained on all data)

**Post-processing:** Binary closing, height-map patching, LUT hole plugging, global fill holes

**References:**
- [1st Place Solution](https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection/writeups/1st-place-solution-for-the-vesuvius-challenge-su)
- [nnU-Net v2](https://github.com/MIC-DKFZ/nnUNet)

In [ ]:
import os
import sys
import csv
import json
import time
import shutil
import zipfile

import numpy as np
import torch
import tifffile

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Configuration

In [ ]:
# --- Paths ---
DATA_DIR = '/kaggle/input/vesuvius-challenge-surface-detection'

# nnU-Net model directory (uploaded as Kaggle Model)
# Update this path to match your uploaded model's slug
MODEL_DIR = '/kaggle/input/models/briansheppard/vesuvius-nnunet/pytorch/default/1'

# --- Inference settings ---
USE_TTA = True              # Test-time augmentation (mirroring)
STEP_SIZE = 0.5             # Sliding window overlap (0.5 = 50% overlap)
USE_POSTPROCESS = True

# --- Post-processing (1st place solution) ---
MIN_COMPONENT_SIZE = 500
CLOSING_RADIUS = 3
ENABLE_PATCHING = True
ENABLE_HOLE_PLUGGING = True

# --- Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# --- nnU-Net environment ---
os.environ['nnUNet_raw'] = '/kaggle/working/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/kaggle/working/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/kaggle/working/nnUNet_results'
for d in ['/kaggle/working/nnUNet_raw', '/kaggle/working/nnUNet_preprocessed', '/kaggle/working/nnUNet_results']:
    os.makedirs(d, exist_ok=True)

# Verify model exists
for f in ['plans.json', 'dataset.json']:
    p = os.path.join(MODEL_DIR, f)
    print(f'  {f}: {"OK" if os.path.exists(p) else "MISSING"}')

fold_dir = os.path.join(MODEL_DIR, 'fold_all')
ckpt_path = os.path.join(fold_dir, 'checkpoint_final.pth')
print(f'  checkpoint_final.pth: {"OK" if os.path.exists(ckpt_path) else "MISSING"}')

## Load nnU-Net Predictor

In [ ]:
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

predictor = nnUNetPredictor(
    tile_step_size=STEP_SIZE,
    use_gaussian=True,
    use_mirroring=USE_TTA,
    perform_everything_on_device=True,
    device=device,
    verbose=False,
    verbose_preprocessing=False,
    allow_tqdm=True,
)

predictor.initialize_from_trained_model_folder(
    MODEL_DIR,
    use_folds=['all'],
    checkpoint_name='checkpoint_final.pth',
)

print(f'nnU-Net predictor initialized')
print(f'  Patch size: {predictor.configuration_manager.patch_size}')
print(f'  TTA (mirroring): {USE_TTA}')
print(f'  Step size: {STEP_SIZE}')

## Post-Processing (1st Place Solution)

Adapted from the 1st place solution: binary closing, height-map patching, LUT-based hole plugging, global fill holes.

In [ ]:
from scipy import ndimage
from scipy.ndimage import (
    binary_closing,
    binary_fill_holes,
    distance_transform_edt,
    find_objects,
    generate_binary_structure,
    label as ndimage_label,
)


def _get_structure(connectivity):
    if connectivity == 6:
        return generate_binary_structure(3, 1)
    elif connectivity == 18:
        return generate_binary_structure(3, 2)
    else:
        return generate_binary_structure(3, 3)


def _pad_slices(sl, shape, pad):
    return tuple(
        slice(max(0, s.start - pad), min(dim, s.stop + pad))
        for s, dim in zip(sl, shape)
    )


def remove_small_components(mask, min_size=500, connectivity=26):
    struct = _get_structure(connectivity)
    labeled, n = ndimage_label(mask, structure=struct)
    if n == 0:
        return mask
    sizes = ndimage.sum(mask, labeled, range(1, n + 1))
    keep = np.zeros_like(mask)
    for i, size in enumerate(sizes, 1):
        if size >= min_size:
            keep[labeled == i] = 1
    return keep.astype(np.uint8)


def make_ball_footprint(radius):
    zz, yy, xx = np.ogrid[
        -radius:radius + 1,
        -radius:radius + 1,
        -radius:radius + 1,
    ]
    return (zz ** 2 + yy ** 2 + xx ** 2) <= radius ** 2


def _count_internal_holes(mask):
    inv = 1 - mask.astype(np.uint8)
    struct6 = generate_binary_structure(3, 1)
    labeled, n = ndimage_label(inv, structure=struct6)
    if n == 0:
        return 0
    border = np.zeros(n + 1, dtype=bool)
    border[labeled[0]] = True
    border[labeled[-1]] = True
    border[labeled[:, 0]] = True
    border[labeled[:, -1]] = True
    border[labeled[:, :, 0]] = True
    border[labeled[:, :, -1]] = True
    border[0] = True
    return int(n - border[1:].sum())


def _height_map_patch_crop(crop):
    if crop.sum() == 0:
        return crop
    best_axis, best_area = 0, 0
    for axis in range(3):
        area = crop.max(axis=axis).sum()
        if area > best_area:
            best_area = area
            best_axis = axis
    crop_t = np.moveaxis(crop, best_axis, 0)
    D, H, W = crop_t.shape
    depth_coords = np.arange(D, dtype=np.float32).reshape(D, 1, 1)
    valid_3d = crop_t.astype(bool)
    count_map = valid_3d.sum(axis=0)
    has_voxels = count_map > 0
    height_map = np.full((H, W), np.nan, dtype=np.float32)
    thick_map = np.zeros((H, W), dtype=np.float32)
    depth_sum = (depth_coords * valid_3d).sum(axis=0)
    height_map[has_voxels] = depth_sum[has_voxels] / count_map[has_voxels]
    thick_map[has_voxels] = count_map[has_voxels]
    filled_proj = binary_fill_holes(has_voxels)
    gap_mask = filled_proj & ~has_voxels
    if not gap_mask.any():
        return crop
    holes_before = _count_internal_holes(crop)
    fill_row_h = np.full((H, W), np.nan, dtype=np.float32)
    fill_row_t = np.full((H, W), np.nan, dtype=np.float32)
    for r in range(H):
        valid_cols = np.where(has_voxels[r])[0]
        gap_cols = np.where(gap_mask[r])[0]
        if len(valid_cols) >= 2 and len(gap_cols) > 0:
            fill_row_h[r, gap_cols] = np.interp(gap_cols, valid_cols, height_map[r, valid_cols])
            fill_row_t[r, gap_cols] = np.interp(gap_cols, valid_cols, thick_map[r, valid_cols])
    fill_col_h = np.full((H, W), np.nan, dtype=np.float32)
    fill_col_t = np.full((H, W), np.nan, dtype=np.float32)
    for c in range(W):
        valid_rows = np.where(has_voxels[:, c])[0]
        gap_rows = np.where(gap_mask[:, c])[0]
        if len(valid_rows) >= 2 and len(gap_rows) > 0:
            fill_col_h[gap_rows, c] = np.interp(gap_rows, valid_rows, height_map[valid_rows, c])
            fill_col_t[gap_rows, c] = np.interp(gap_rows, valid_rows, thick_map[valid_rows, c])
    not_valid = ~has_voxels
    row_dist = distance_transform_edt(not_valid, sampling=[1e6, 1])
    col_dist = distance_transform_edt(not_valid, sampling=[1, 1e6])
    gap_r, gap_c = np.where(gap_mask)
    hr, hc = fill_row_h[gap_r, gap_c], fill_col_h[gap_r, gap_c]
    tr, tc = fill_row_t[gap_r, gap_c], fill_col_t[gap_r, gap_c]
    dr = np.maximum(row_dist[gap_r, gap_c], 1e-6)
    dc = np.maximum(col_dist[gap_r, gap_c], 1e-6)
    valid_r, valid_c = ~np.isnan(hr), ~np.isnan(hc)
    both = valid_r & valid_c
    only_r = valid_r & ~valid_c
    only_c = valid_c & ~valid_r
    wr = np.where(both, 1.0 / dr, 0.0)
    wc = np.where(both, 1.0 / dc, 0.0)
    w_total = np.maximum(wr + wc, 1e-12)
    h_avg = np.where(both, (np.nan_to_num(hr) * wr + np.nan_to_num(hc) * wc) / w_total,
                     np.where(only_r, hr, np.where(only_c, hc, np.nan)))
    t_avg = np.where(both, (np.nan_to_num(tr) * wr + np.nan_to_num(tc) * wc) / w_total,
                     np.where(only_r, tr, np.where(only_c, tc, 0.0)))
    patched_t = crop_t.copy()
    for idx in range(len(gap_r)):
        h_val = h_avg[idx]
        if np.isnan(h_val):
            continue
        r, c = gap_r[idx], gap_c[idx]
        center = int(round(h_val))
        half = max(0, int(round(t_avg[idx] / 2)))
        z0, z1 = max(0, center - half), min(D - 1, center + half)
        patched_t[z0:z1 + 1, r, c] = 1
    patched_3d = np.moveaxis(patched_t, 0, best_axis)
    if _count_internal_holes(patched_3d) > holes_before:
        return crop
    return patched_3d.astype(np.uint8)


def _build_hole_plug_lut():
    face_diags = [
        (0, 3, 1, 2), (1, 2, 0, 3), (4, 7, 5, 6), (5, 6, 4, 7),
        (0, 5, 1, 4), (1, 4, 0, 5), (2, 7, 3, 6), (3, 6, 2, 7),
        (0, 6, 2, 4), (2, 4, 0, 6), (1, 7, 3, 5), (3, 5, 1, 7),
    ]
    lut = np.zeros(256, dtype=np.uint8)
    for pattern in range(256):
        add = 0
        for fa, fb, g1, g2 in face_diags:
            if ((pattern >> fa) & 1) and ((pattern >> fb) & 1) \
                    and not ((pattern >> g1) & 1) and not ((pattern >> g2) & 1):
                add |= (1 << g1)
        lut[pattern] = add
    return lut

_HOLE_PLUG_LUT = None

def plug_holes_lut(mask, max_iterations=5):
    global _HOLE_PLUG_LUT
    if _HOLE_PLUG_LUT is None:
        _HOLE_PLUG_LUT = _build_hole_plug_lut()
    lut = _HOLE_PLUG_LUT
    result = mask.astype(np.uint8).copy()
    Z, Y, X = result.shape
    if Z < 2 or Y < 2 or X < 2:
        return result
    offsets = [(dz, dy, dx) for dz in range(2) for dy in range(2) for dx in range(2)]
    for _ in range(max_iterations):
        pattern = np.zeros((Z - 1, Y - 1, X - 1), dtype=np.uint8)
        for bit, (dz, dy, dx) in enumerate(offsets):
            pattern |= (result[dz:Z-1+dz, dy:Y-1+dy, dx:X-1+dx] << bit)
        additions = lut[pattern]
        if not additions.any():
            break
        for bit, (dz, dy, dx) in enumerate(offsets):
            result[dz:Z-1+dz, dy:Y-1+dy, dx:X-1+dx] |= ((additions >> bit) & 1).astype(np.uint8)
    return result


def postprocess(pred, min_component_size=500, closing_radius=3,
                enable_patching=True, enable_hole_plugging=True, connectivity=26):
    """Full 1st place post-processing pipeline."""
    t_total = time.time()
    pred = pred.astype(np.uint8)
    struct = _get_structure(connectivity)

    t0 = time.time()
    pred = remove_small_components(pred, min_component_size, connectivity)
    print(f'  Step 1 (CC filter): {time.time() - t0:.2f}s')

    labeled, n = ndimage_label(pred, structure=struct)
    if n == 0:
        return pred

    slices = find_objects(labeled)
    footprint = make_ball_footprint(closing_radius) if closing_radius > 0 else None
    pad = closing_radius if closing_radius > 0 else 0
    result = np.zeros_like(pred, dtype=np.uint8)

    t_close = time.time()
    t_patch, t_plug = 0.0, 0.0

    for comp_id, sl in enumerate(slices, 1):
        if sl is None:
            continue
        padded_sl = _pad_slices(sl, pred.shape, pad)
        crop = (labeled[padded_sl] == comp_id).astype(np.uint8)
        if closing_radius > 0:
            crop = binary_closing(crop, structure=footprint).astype(np.uint8)
        if enable_patching:
            tp = time.time()
            crop = _height_map_patch_crop(crop)
            t_patch += time.time() - tp
        if enable_hole_plugging:
            tg = time.time()
            crop = plug_holes_lut(crop)
            t_plug += time.time() - tg
        result[padded_sl] |= crop

    print(f'  Step 2 (closing): {time.time() - t_close - t_patch - t_plug:.2f}s')
    print(f'  Step 3 (patching): {t_patch:.2f}s')
    print(f'  Step 4 (hole plug): {t_plug:.2f}s')

    t0 = time.time()
    result = binary_fill_holes(result).astype(np.uint8)
    print(f'  Step 5 (fill holes): {time.time() - t0:.2f}s')
    print(f'  Post-processing total: {time.time() - t_total:.2f}s')
    return result

print('Post-processing functions loaded.')

## Prepare Test Data for nnU-Net

nnU-Net expects test images in the same format as training: `{case_id}_0000.tif` with a spacing JSON.

In [ ]:
with open(os.path.join(DATA_DIR, 'test.csv')) as f:
    test_ids = [row['id'] for row in csv.DictReader(f)]
print(f'Test IDs: {test_ids}')

VOXEL_SPACING = 7.91
test_input_dir = '/kaggle/working/test_nnunet'
os.makedirs(test_input_dir, exist_ok=True)

for test_id in test_ids:
    src = os.path.join(DATA_DIR, 'test_images', f'{test_id}.tif')
    dst = os.path.join(test_input_dir, f'{test_id}_0000.tif')
    if not os.path.exists(dst):
        os.symlink(os.path.abspath(src), dst)
    json_path = os.path.join(test_input_dir, f'{test_id}.json')
    with open(json_path, 'w') as jf:
        json.dump({'spacing': [VOXEL_SPACING, VOXEL_SPACING, VOXEL_SPACING]}, jf)
    print(f'  {test_id}: linked + spacing JSON')

print(f'Test data prepared in {test_input_dir}')

## Run Inference

In [ ]:
t_start = time.time()

raw_output_dir = '/kaggle/working/nnunet_raw_predictions'
os.makedirs(raw_output_dir, exist_ok=True)

print('Running nnU-Net inference...')
predictor.predict_from_files(
    test_input_dir,
    raw_output_dir,
    save_probabilities=False,
    overwrite=True,
    num_processes_preprocessing=2,
    num_processes_segmentation_export=2,
)

print(f'\nInference complete in {time.time() - t_start:.1f}s')
print(f'Raw predictions saved to {raw_output_dir}')
for f in sorted(os.listdir(raw_output_dir)):
    if f.endswith('.tif'):
        print(f'  {f}')

## Post-Process and Create Submission

In [ ]:
os.makedirs('/kaggle/working/submission_files', exist_ok=True)
predictions = {}

for test_id in test_ids:
    print(f'\n=== {test_id} ===')
    pred_path = os.path.join(raw_output_dir, f'{test_id}.tif')
    pred = tifffile.imread(pred_path)
    print(f'nnU-Net prediction shape: {pred.shape}, unique labels: {np.unique(pred)}')

    # Extract surface mask (label 1)
    surface_mask = (pred == 1).astype(np.uint8)
    print(f'Surface voxels: {surface_mask.sum()} ({surface_mask.sum()/surface_mask.size*100:.1f}%)')

    if USE_POSTPROCESS:
        print('Post-processing (1st place pipeline)...')
        surface_mask = postprocess(
            surface_mask, MIN_COMPONENT_SIZE, CLOSING_RADIUS,
            ENABLE_PATCHING, ENABLE_HOLE_PLUGGING
        )
        print(f'After post-processing: {surface_mask.sum()} voxels ({surface_mask.sum()/surface_mask.size*100:.1f}%)')

    predictions[test_id] = surface_mask

print(f'\nTotal time so far: {time.time() - t_start:.1f}s')

In [ ]:
for test_id, pred in predictions.items():
    tif_path = f'/kaggle/working/submission_files/{test_id}.tif'
    tifffile.imwrite(tif_path, pred)
    print(f'Saved {tif_path}: shape={pred.shape}, dtype={pred.dtype}')

with zipfile.ZipFile('/kaggle/working/submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for test_id in predictions:
        tif_path = f'/kaggle/working/submission_files/{test_id}.tif'
        zf.write(tif_path, f'{test_id}.tif')

print(f'\nSubmission saved to /kaggle/working/submission.zip')
print(f'Total time: {time.time() - t_start:.1f}s')